# Lab | Multi-Agent Bidding

# Multi-Agent Decentralized Speaker Selection

This notebook showcases how to implement a multi-agent simulation without a fixed schedule for who speaks when. Instead the agents decide for themselves who speaks. We implement this by having each agent **bid** to speak — whichever agent bids the highest gets the floor.

The example below simulates a fictitious presidential debate using **Anthropic Claude** via LangChain.

In [1]:
!pip install langchain-openai tenacity numpy -q


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [2]:
import os
import re
import numpy as np
from typing import Callable, List

import tenacity
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

## `DialogueAgent` and `DialogueSimulator` classes

The `DialogueAgent` wraps an LLM and maintains its own message history.
The `DialogueSimulator` orchestrates the conversation by repeatedly:
1. Selecting the next speaker via a selection function
2. Having that agent send a message
3. Distributing the message to all agents

In [4]:
class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
    ) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        message = self.model.invoke(
            [
                self.system_message,
                HumanMessage(content="\n".join(self.message_history + [self.prefix])),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        self.message_history.append(f"{name}: {message}")


class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        for agent in self.agents:
            agent.receive(name, message)
        self._step += 1

    def step(self) -> tuple[str, str]:
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]
        message = speaker.send()
        for receiver in self.agents:
            receiver.receive(speaker.name, message)
        self._step += 1
        return speaker.name, message

## `BiddingDialogueAgent` class

A subclass of `DialogueAgent` that adds a `bid()` method. The agent rates how strongly it wants to respond to the most recent message, returning an integer.

In [5]:
class BiddingDialogueAgent(DialogueAgent):
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        bidding_template: str,
        model: ChatOpenAI,
    ) -> None:
        super().__init__(name, system_message, model)
        self.bidding_template = bidding_template

    def bid(self) -> str:
        prompt = PromptTemplate(
            input_variables=["message_history", "recent_message"],
            template=self.bidding_template,
        ).format(
            message_history="\n".join(self.message_history),
            recent_message=self.message_history[-1],
        )
        bid_string = self.model.invoke([HumanMessage(content=prompt)]).content
        return bid_string

## Define participants and debate topic

In [6]:
character_names = ["Donald Trump", "Kanye West", "Elizabeth Warren"]
topic = "transcontinental high speed rail"
word_limit = 50

## Generate system messages

For each candidate we generate:
1. A creative character description
2. A character header (context block)
3. A system message used to prime the agent's LLM

In [13]:
topic_specifier_prompt = [
    SystemMessage(content="You can make a task more specific."),
    HumanMessage(
        content=(
            f"{game_description}\n\n"
            f"You are the debate moderator.\n"
            f"Please make the debate topic more specific.\n"
            f"Frame the debate topic as a problem to be solved.\n"
            f"Be creative and imaginative.\n"
            f"Please reply with the specified topic in {word_limit} words or less.\n"
            f"Speak directly to the presidential candidates: {', '.join(character_names)}.\n"
            f"Do not add anything else."
        )
    ),
]
specified_topic = ChatOpenAI(temperature=1.0).invoke(topic_specifier_prompt).content

print(f"Original topic:\n{topic}\n")
print(f"Detailed topic:\n{specified_topic}\n")

Original topic:
transcontinental high speed rail

Detailed topic:
Candidates, the specific debate topic is: How can we collaborate to implement a nationwide transcontinental high-speed rail system that connects major cities, reduces carbon emissions, creates jobs, and improves transportation infrastructure for the long-term benefit of the American people?



In [14]:
for name, desc, header, sys_msg in zip(
    character_names,
    character_descriptions,
    character_headers,
    character_system_messages,
):
    print(f"\n{'='*60}")
    print(f"{name} Description:")
    print(desc)
    print(f"\nCharacter Header:")
    print(header)
    print(f"\nSystem Message:")
    print(sys_msg.content)


Donald Trump Description:
Donald Trump, you are known for your bold and brash personality, unafraid to speak your mind and take controversial stances. You exude confidence and have a knack for making headlines with your strong opinions. Your no-nonsense approach resonates with many, while also drawing criticism.

Character Header:
Here is the topic for the presidential debate: transcontinental high speed rail.
The presidential candidates are: Donald Trump, Kanye West, Elizabeth Warren.
Your name is Donald Trump.
You are a presidential candidate.
Your description is as follows: Donald Trump, you are known for your bold and brash personality, unafraid to speak your mind and take controversial stances. You exude confidence and have a knack for making headlines with your strong opinions. Your no-nonsense approach resonates with many, while also drawing criticism.
You are debating the topic: transcontinental high speed rail.
Your goal is to be as creative as possible and make the voters th

## Output parser for bids

Agents output a bid as an integer wrapped in angle brackets, e.g. `<7>`. We implement a custom `BaseOutputParser` to parse this format reliably.

In [15]:
class BidOutputParser(BaseOutputParser):
    """Parses a bid integer enclosed in angle brackets: <int>."""

    def get_format_instructions(self) -> str:
        return "Your response should be an integer delimited by angled brackets, like this: <int>."

    def parse(self, text: str) -> dict:
        match = re.search(r"<(\d+)>", text)
        if not match:
            raise ValueError(f"Could not parse bid from: {text!r}")
        return {"bid": match.group(1)}

    @property
    def _type(self) -> str:
        return "bid_output_parser"


bid_parser = BidOutputParser()
print("Format instructions:", bid_parser.get_format_instructions())

Format instructions: Your response should be an integer delimited by angled brackets, like this: <int>.


## Generate bidding templates

Each agent gets a bidding prompt that asks it to rate (1–10) how contradictory the most recent message is to its own ideas. A higher contradiction score means the agent wants to respond more urgently.

In [16]:
def generate_character_bidding_template(character_header: str) -> str:
    return (
        f"{character_header}\n\n"
        f"```\n{{message_history}}\n```\n\n"
        f"On the scale of 1 to 10, where 1 is not contradictory and 10 is extremely contradictory, "
        f"rate how contradictory the following message is to your ideas.\n\n"
        f"```\n{{recent_message}}\n```\n\n"
        f"{bid_parser.get_format_instructions()}\n"
        f"Do nothing else."
    )


character_bidding_templates = [
    generate_character_bidding_template(header)
    for header in character_headers
]

In [17]:
for name, bidding_template in zip(character_names, character_bidding_templates):
    print(f"{name} Bidding Template:")
    print(bidding_template)
    print()

Donald Trump Bidding Template:
Here is the topic for the presidential debate: transcontinental high speed rail.
The presidential candidates are: Donald Trump, Kanye West, Elizabeth Warren.
Your name is Donald Trump.
You are a presidential candidate.
Your description is as follows: Donald Trump, you are known for your bold and brash personality, unafraid to speak your mind and take controversial stances. You exude confidence and have a knack for making headlines with your strong opinions. Your no-nonsense approach resonates with many, while also drawing criticism.
You are debating the topic: transcontinental high speed rail.
Your goal is to be as creative as possible and make the voters think you are the best candidate.


```
{message_history}
```

On the scale of 1 to 10, where 1 is not contradictory and 10 is extremely contradictory, rate how contradictory the following message is to your ideas.

```
{recent_message}
```

Your response should be an integer delimited by angled brackets

## Use an LLM to elaborate on the debate topic

In [19]:
topic_specifier_prompt = [
    SystemMessage(content="You can make a task more specific."),
    HumanMessage(
        content=(
            f"{game_description}\n\n"
            f"You are the debate moderator.\n"
            f"Please make the debate topic more specific.\n"
            f"Frame the debate topic as a problem to be solved.\n"
            f"Be creative and imaginative.\n"
            f"Please reply with the specified topic in {word_limit} words or less.\n"
            f"Speak directly to the presidential candidates: {', '.join(character_names)}.\n"
            f"Do not add anything else."
        )
    ),
]
specified_topic = ChatOpenAI(temperature=1.0).invoke(topic_specifier_prompt).content

print(f"Original topic:\n{topic}\n")
print(f"Detailed topic:\n{specified_topic}\n")

Original topic:
transcontinental high speed rail

Detailed topic:
Candidates, the debate topic is: "Developing a Transcontinental High-Speed Rail System by 2040 to revolutionize transportation, reduce greenhouse gas emissions, and stimulate economic growth. What innovative funding solutions and collaboration strategies will each of you propose to ensure the success and efficiency of this ambitious infrastructure project?"



## Define the speaker selection function

Each agent bids to speak. The agent with the highest bid is selected; ties are broken randomly.

We use `tenacity` to retry if the bid cannot be parsed, defaulting to 0 after exhausting retries.

In [20]:
@tenacity.retry(
    stop=tenacity.stop_after_attempt(2),
    wait=tenacity.wait_none(),
    retry=tenacity.retry_if_exception_type(ValueError),
    before_sleep=lambda retry_state: print(
        f"ValueError: {retry_state.outcome.exception()}, retrying..."
    ),
    retry_error_callback=lambda retry_state: 0,
)
def ask_for_bid(agent: BiddingDialogueAgent) -> int:
    """Ask the agent for a bid and parse it into an integer."""
    bid_string = agent.bid()
    bid = int(bid_parser.parse(bid_string)["bid"])
    return bid

In [21]:
def select_next_speaker(step: int, agents: List[BiddingDialogueAgent]) -> int:
    bids = [ask_for_bid(agent) for agent in agents]

    # randomly select among agents with the same highest bid
    max_value = np.max(bids)
    max_indices = np.where(np.array(bids) == max_value)[0]
    idx = int(np.random.choice(max_indices))

    print("Bids:")
    for i, (bid, agent) in enumerate(zip(bids, agents)):
        marker = " <-- selected" if i == idx else ""
        print(f"  {agent.name}: {bid}{marker}")
    print()
    return idx

## Main loop

Instantiate the agents and run the debate for `max_iters` turns.

In [23]:
characters = [
    BiddingDialogueAgent(
        name=name,
        system_message=sys_msg,
        model=ChatOpenAI(temperature=0.2),
        bidding_template=bidding_template,
    )
    for name, sys_msg, bidding_template in zip(
        character_names, character_system_messages, character_bidding_templates
    )
]

In [24]:
max_iters = 10
n = 0

simulator = DialogueSimulator(agents=characters, selection_function=select_next_speaker)
simulator.reset()
simulator.inject("Debate Moderator", specified_topic)
print(f"(Debate Moderator): {specified_topic}\n")

while n < max_iters:
    name, message = simulator.step()
    print(f"({name}): {message}\n")
    n += 1

(Debate Moderator): Candidates, the debate topic is: "Developing a Transcontinental High-Speed Rail System by 2040 to revolutionize transportation, reduce greenhouse gas emissions, and stimulate economic growth. What innovative funding solutions and collaboration strategies will each of you propose to ensure the success and efficiency of this ambitious infrastructure project?"

Bids:
  Donald Trump: 8 <-- selected
  Kanye West: 1
  Elizabeth Warren: 1

(Donald Trump): Let me tell you, folks, when it comes to high-speed rail, I have the best ideas. We're going to build the most luxurious trains you've ever seen, with gold-plated tracks and crystal chandeliers in every carriage. *I'll make high-speed rail great again!*

Bids:
  Donald Trump: 7
  Kanye West: 9 <-- selected
  Elizabeth Warren: 9

(Kanye West): *As I step onto the stage, my presence commands attention. High-speed rail isn't just about transportation; it's a canvas for creativity. Picture this: trains designed by the world's